In [1]:
from datetime import date
from pathlib import Path

import numpy as np
import pandas as pd

In [6]:
print("cwd:", Path.cwd())
DATA_DIR = Path("Datos") 
print("Datos existe:", DATA_DIR.exists())
print("ficheros:", sorted(p.name for p in DATA_DIR.glob("penguins.*")))

cwd: /Users/juandiicai/Desktop/2 - MASTER/app_visualizacion_datos_2026
Datos existe: True
ficheros: ['penguins.csv']


In [8]:
df = pd.read_csv(DATA_DIR / "penguins.csv")
df.head()

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,MALE
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,FEMALE
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,FEMALE
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,FEMALE


In [20]:
df.shape
df.columns



Index(['species', 'island', 'bill_length_mm', 'bill_depth_mm',
       'flipper_length_mm', 'body_mass_g', 'sex'],
      dtype='object')

In [12]:
df.dtypes

species               object
island                object
bill_length_mm       float64
bill_depth_mm        float64
flipper_length_mm    float64
body_mass_g          float64
sex                   object
dtype: object

In [18]:
df.isna().sum()

species               0
island                0
bill_length_mm        2
bill_depth_mm         2
flipper_length_mm     2
body_mass_g           2
sex                  11
dtype: int64

In [34]:
def validar_penguins(frame: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    df = frame.copy()

    columnas_numericas = [
        "bill_length_mm",
        "bill_depth_mm",
        "flipper_length_mm",
        "body_mass_g",
    ]

    for columna in columnas_numericas:
        df[columna] = pd.to_numeric(df[columna], errors="coerce")

    df["sex"] = df["sex"].str.upper()

    mascara_validos = (
        df[columnas_numericas].notna().all(axis=1)
        & df[columnas_numericas].gt(0).all(axis=1)
        & df["sex"].isin(["MALE", "FEMALE"])
        & df["species"].notna()
        & df["island"].notna()
    )

    validos = df.loc[mascara_validos].copy()
    errores = df.loc[~mascara_validos].copy()

    return validos, errores
        
    

In [43]:
validos, errores = validar_penguins(df)

In [44]:
masa_media_por_especie = (
    validos
    .groupby("species", as_index=False)
    .agg(masa_media=("body_mass_g", "mean"))
)

print(masa_media_por_especie)

     species   masa_media
0     Adelie  3706.164384
1  Chinstrap  3733.088235
2     Gentoo  5092.436975


In [45]:
count_island = (
    validos
    .groupby("island", as_index=False)
    .agg(count_island=("body_mass_g", "count"))
)

print(count_island)





      island  count_island
0     Biscoe           163
1      Dream           123
2  Torgersen            47


In [46]:
count_sex = (
    validos
    .groupby("sex", as_index=False)
    .agg(count_sex=("body_mass_g", "count"))
)

print(count_sex)

      sex  count_sex
0  FEMALE        165
1    MALE        168


In [47]:
validos.to_csv("Datos/penguins_limpios.csv", index=False)